# Tuần 10 — So sánh B0, B1, B2 và Proposed (YOLOv8s) — Dual-Mode (Local & Kaggle)

Đánh giá trên cùng test split đã khóa (480 ảnh) của PKU-Market-PCB:
- **B0**: YOLOv8s baseline (ngưỡng mặc định 0.25)
- **B1**: YOLOv8s-P2 (ngưỡng mặc định 0.25)
- **B2**: YOLOv8s baseline + Dual Threshold tối ưu (tau_crit=0.02, tau_reg=0.58)
- **Proposed**: YOLOv8s-P2 + Dual Threshold tối ưu (tau_crit=0.01, tau_reg=0.57)

Hỗ trợ chạy mượt mà cả trên **Local Mac (Apple MPS GPU)** lẫn **Kaggle GPU (Tesla T4)**.


## 1. Khởi tạo và xác thực split/checkpoint

In [1]:
# ── 1. Môi trường & Cài đặt (Tự động nhận diện Kaggle vs Local Mac) ─────────
import gc, hashlib, json, os, platform, shutil, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import yaml
from IPython.display import display, FileLink

# Dọn sạch VRAM còn sót lại từ session trước và chống phân mảnh bộ nhớ
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    # Tắt benchmark mode để cuDNN không thử cấp phát workspace khổng lồ (7.5GB - 15GB)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

IS_KAGGLE = Path('/kaggle').exists()
if IS_KAGGLE:
    import subprocess
    print('🚀 Đang chạy trên Kaggle GPU — Cài đặt môi trường...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'ray', '-y', '-q'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.3.40', '-q'])

from ultralytics import YOLO

CLASSES = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']
CRITICAL = {2, 3}  # Open_circuit, Short
IMGSZ, PRED_CONF, NMS_IOU, MATCH_IOU, DEFAULT_CONF = 1024, 0.001, 0.7, 0.50, 0.25

if IS_KAGGLE:
    ROOT = Path('/kaggle/working')
    os.chdir(ROOT)
    input_root = Path('/kaggle/input')
    
    # 1. Tìm và giải nén hoặc đồng bộ bundle pcb_compare_yolov8s_bundle
    zip_cands = list(input_root.rglob('pcb_compare_yolov8s_bundle.zip'))
    if zip_cands:
        with zipfile.ZipFile(zip_cands[0]) as zf:
            zf.extractall(ROOT)
        bundle_dir = ROOT / 'pcb_compare_yolov8s_bundle'
        if bundle_dir.is_dir():
            for item in bundle_dir.iterdir():
                dest = ROOT / item.name
                if dest.exists():
                    if dest.is_dir(): shutil.rmtree(dest)
                    else: dest.unlink()
                shutil.move(str(item), str(ROOT))
        print(f'✅ Đã giải nén bundle từ: {zip_cands[0]}')
    else:
        thresh_files = list(input_root.rglob('threshold_yolov8s.yaml'))
        assert thresh_files, 'Không tìm thấy bundle! Hãy Add Input dataset chứa pcb_compare_yolov8s_bundle.'
        bundle_root = thresh_files[0].parent.parent
        print(f'✅ Tìm thấy bundle folder: {bundle_root}')
        for sub in ['configs', 'data', 'runs', 'evaluation']:
            src = bundle_root / sub
            dst = ROOT / sub
            if src.exists() and not dst.exists():
                shutil.copytree(src, dst)
        print('✅ Đã đồng bộ cấu hình và checkpoints vào /kaggle/working.')
        
    # 2. Tìm dataset ảnh PKU
    pku_cands = [p for p in input_root.rglob('PKU-Market-PCB(Data enhanced version)') if (p/'test/images').is_dir()]
    assert pku_cands, 'Không tìm thấy dataset ảnh PKU! Hãy Add Input: liuxiaolong1/pcb-defect-detection-dataset.'
    DATA_ROOT = pku_cands[0]
    print(f'✅ Dataset ảnh PKU: {DATA_ROOT}')
    
    DEVICE = 0
    # Batch 2 đảm bảo dùng < 3GB VRAM trên Tesla T4 16GB, tốc độ 480 ảnh chỉ mất ~10-15s
    BATCH = 2
else:
    cwd = Path.cwd().resolve()
    ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'configs/threshold_yolov8s.yaml').is_file()), None)
    assert ROOT is not None, 'Mở notebook từ project PCB-DetectionImprove hoặc scripts/.'
    os.chdir(ROOT)
    DATA_ROOT = ROOT / 'data/raw/PKU-Market-PCB(Data enhanced version)'
    if torch.cuda.is_available(): DEVICE = 0; BATCH = 2; print('CUDA GPU')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available(): DEVICE = 'mps'; BATCH = 4; print('Apple MPS (Apple Silicon GPU)')
    else: DEVICE = 'cpu'; BATCH = 2; print('CPU')

SPLIT = ROOT / 'data/splits/test.txt'
THRESHOLD_FILE = ROOT / 'configs/threshold_yolov8s.yaml'
OUT = ROOT / 'evaluation/B0_B1_yolov8s_week10'
OUT.mkdir(parents=True, exist_ok=True)

config = yaml.safe_load(THRESHOLD_FILE.read_text(encoding='utf-8'))
assert config['experiment_family'] == 'YOLOv8s' and config['test_not_used_for_threshold'] is True
models_cfg = config['models']
CHECKPOINTS = {k: ROOT / v['checkpoint'] for k, v in models_cfg.items()}
for name, path in CHECKPOINTS.items():
    assert path.is_file(), f'{name} checkpoint missing: {path}'
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == models_cfg[name]['checkpoint_sha256'], f'{name} checkpoint hash differs from locked config.'

lines = [s.strip() for s in SPLIT.read_text(encoding='utf-8').splitlines() if s.strip()]
assert len(lines) == 480, f'Expected locked test split of 480 images, found {len(lines)}.'

LOCAL_PREFIX = 'data/raw/PKU-Market-PCB(Data enhanced version)/'
alt_local_dirs = [
    Path('/Users/duytan/orca/projects/PCB-Detect/PKU-Market-PCB(Data enhanced version)'),
    DATA_ROOT
]

def resolve_image(s):
    p = Path(s)
    if p.is_file(): return p.resolve()
    if (ROOT / s).is_file(): return (ROOT / s).resolve()
    rel = s[len(LOCAL_PREFIX):] if s.startswith(LOCAL_PREFIX) else s
    for alt in alt_local_dirs:
        if (alt / rel).is_file(): return (alt / rel).resolve()
    return (DATA_ROOT / rel).resolve()

IMAGES = [resolve_image(s) for s in lines]
assert all(p.is_file() for p in IMAGES), 'Missing test image(s); stop without evaluating.'

# Sinh eval_test_data.yaml chuẩn cho model.val() ở Cell 8 (đảm bảo tuyệt đối không lỗi relative path)
EVAL_SPLIT_FILE = OUT / 'test_eval_split.txt'
with open(EVAL_SPLIT_FILE, 'w', encoding='utf-8') as f:
    for p in IMAGES:
        f.write(str(p.resolve()) + chr(10))

eval_yaml_data = {
    'path': str(ROOT),
    'train': str(EVAL_SPLIT_FILE.resolve()),
    'val': str(EVAL_SPLIT_FILE.resolve()),
    'test': str(EVAL_SPLIT_FILE.resolve()),
    'nc': len(CLASSES),
    'names': {i: name for i, name in enumerate(CLASSES)}
}
DATA_YAML = OUT / 'eval_test_data.yaml'
DATA_YAML.write_text(yaml.safe_dump(eval_yaml_data, sort_keys=False, allow_unicode=True), encoding='utf-8')

# The earlier B0 test artifact used a different absolute root; compare normalized dataset-relative paths.
old_list = ROOT / 'evaluation/B0_yolov8s/test_eval_split.txt'
def normalized_test_list(path):
    vals = [s.strip() for s in path.read_text(encoding='utf-8').splitlines() if s.strip()]
    marker = 'PKU-Market-PCB(Data enhanced version)/'
    return [s.split(marker, 1)[1] if marker in s else s for s in vals]
if old_list.is_file():
    assert normalized_test_list(SPLIT) == normalized_test_list(old_list), 'Test split differs from earlier B0 evaluation.'

print('✅ Environment:', 'KAGGLE' if IS_KAGGLE else 'LOCAL MAC', '| Device:', DEVICE, '| Batch:', BATCH)
print('✅ Test images:', len(IMAGES), '| Test split SHA256:', hashlib.sha256(SPLIT.read_bytes()).hexdigest()[:8])
print('✅ Prior B0 test artifact exists:', (ROOT / 'evaluation/B0_yolov8s/test_metrics.json').is_file())
print('✅ Locked thresholds:', {k: (v['tau_critical'], v['tau_regular']) for k, v in models_cfg.items()})


🚀 Đang chạy trên Kaggle GPU — Cài đặt môi trường...


✅ Tìm thấy bundle folder: /kaggle/input/datasets/tanthongthai/pcb-compare-yolov8s-bundle/pcb_compare_yolov8s_bundle
✅ Đã đồng bộ cấu hình và checkpoints vào /kaggle/working.
✅ Dataset ảnh PKU: /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)
✅ Environment: KAGGLE | Device: 0 | Batch: 2
✅ Test images: 480 | Test split SHA256: 488eed2b
✅ Prior B0 test artifact exists: True
✅ Locked thresholds: {'B0': (0.02, 0.58), 'B1': (0.01, 0.57)}


## 2. Ground truth, size bins và inference

Size bins dùng quy ước đã ghi trong `docs/01-Reports/decision_log.md`: quy đổi diện tích box về ảnh 640×640; small `<32²`, medium `32²–96²`, large `>96²`. Matching class-aware, one-to-one, IoU ≥ 0.50. Dự đoán được lấy một lần từ mỗi checkpoint ở conf 0.001 rồi áp ngưỡng từng cấu hình offline.

In [2]:
def label_path(image):
    parts = list(image.parts)
    assert 'images' in parts, f'Cannot infer label path: {image}'
    parts[parts.index('images')] = 'labels'
    return Path(*parts).with_suffix('.txt')

def load_gt(image):
    with Image.open(image) as im:
        iw, ih = im.size
    rows = []
    p = label_path(image)
    for line in p.read_text(encoding='utf-8').splitlines():
        vals = line.split()
        if len(vals) < 5:
            continue
        cid, xc, yc, w, h = map(float, vals[:5])
        cid = int(cid)
        if cid not in range(6):
            continue
        box = np.array([(xc-w/2)*iw, (yc-h/2)*ih, (xc+w/2)*iw, (yc+h/2)*ih], dtype=float)
        area640 = w*h*640*640
        size = 'small' if area640 < 32**2 else ('medium' if area640 <= 96**2 else 'large')
        rows.append({'class_id': cid, 'box': box, 'size': size})
    return rows

def iou_one_to_many(box, boxes):
    if len(boxes) == 0:
        return np.empty(0, dtype=float)
    x1, y1 = np.maximum(box[:2], boxes[:, :2]).T
    x2, y2 = np.minimum(box[2:], boxes[:, 2:]).T
    inter = np.maximum(0, x2-x1)*np.maximum(0, y2-y1)
    a = max(0, box[2]-box[0])*max(0, box[3]-box[1])
    b = np.maximum(0, boxes[:,2]-boxes[:,0])*np.maximum(0, boxes[:,3]-boxes[:,1])
    return inter/np.maximum(a+b-inter, 1e-12)

GT = [load_gt(p) for p in IMAGES]
assert all(label_path(p).is_file() for p in IMAGES), 'Missing label file(s); stop.'
print('GT boxes:', sum(map(len, GT)), '| clean images:', sum(not x for x in GT))

records = {}
for name, checkpoint in CHECKPOINTS.items():
    print('Inference:', name, checkpoint)
    model = YOLO(str(checkpoint))
    t0 = time.time()
    model_records = []
    # ĐẶC BIỆT QUAN TRỌNG: Truyền source là đường dẫn file .txt (EVAL_SPLIT_FILE)
    # Nếu truyền Python list (source=[...]), Ultralytics sẽ ép toàn bộ 480 ảnh thành 1 batch gây tràn 15GB VRAM!
    with torch.no_grad():
        for result in model.predict(source=str(EVAL_SPLIT_FILE), imgsz=IMGSZ, batch=BATCH,
                                    device=DEVICE, conf=PRED_CONF, iou=NMS_IOU, max_det=300,
                                    stream=True, verbose=False):
            if result.boxes is None or len(result.boxes) == 0:
                model_records.append([])
                continue
            boxes = result.boxes.xyxy.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)
            scores = result.boxes.conf.cpu().numpy()
            model_records.append([{'box': b, 'class_id': int(c), 'score': float(s)} for b,c,s in zip(boxes,classes,scores)])
    assert len(model_records) == len(IMAGES), f'Số kết quả ({len(model_records)}) không khớp số ảnh ({len(IMAGES)})'
    records[name] = model_records
    print(f'  {len(model_records)} images in {time.time()-t0:.1f}s')
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()


GT boxes: 2424 | clean images: 0
Inference: B0 /kaggle/working/runs/B0_yolov8s_100ep_batch16/weights/best.pt
  480 images in 36.8s
Inference: B1 /kaggle/working/runs/B1_yolov8s_p2_100ep/weights/best.pt
  480 images in 37.1s


## 3. Metric tại mỗi cấu hình

B0/B1 dùng conf 0.25 cho cả lớp. B2/Proposed dùng đúng `tau_critical` cho Open_circuit/Short và `tau_regular` cho bốn lớp còn lại. Board-level: ảnh có ít nhất một detection sau threshold là DEFECTIVE; nếu không là PASS. Do test split không có ảnh sạch, tỷ lệ clean→DEFECTIVE sẽ là N/A.

In [3]:
def evaluate_fixed_threshold(model_name, model_records, tau_critical, tau_regular, config_name):
    tp = np.zeros(6, dtype=int); fp = np.zeros(6, dtype=int); gt_count = np.zeros(6, dtype=int)
    size_gt = {s: 0 for s in ('small','medium','large')}
    size_tp = {s: 0 for s in ('small','medium','large')}
    defective = np.array([len(g)>0 for g in GT]); predicted_defective = np.zeros(len(GT), dtype=bool)
    for image_idx, (gt, dets) in enumerate(zip(GT, model_records)):
        for g in gt:
            gt_count[g['class_id']] += 1; size_gt[g['size']] += 1
        kept = [d for d in dets if d['score'] >= (tau_critical if d['class_id'] in CRITICAL else tau_regular)]
        predicted_defective[image_idx] = bool(kept)
        matched = set()
        for d in sorted(kept, key=lambda x: -x['score']):
            cid = d['class_id']
            candidates = [(j,g) for j,g in enumerate(gt) if g['class_id']==cid and j not in matched]
            if candidates:
                ious = iou_one_to_many(d['box'], np.stack([g['box'] for _,g in candidates]))
                best = int(np.argmax(ious))
                if ious[best] >= MATCH_IOU:
                    j,g = candidates[best]; matched.add(j); tp[cid] += 1; size_tp[g['size']] += 1
                    continue
            fp[cid] += 1
    fn = gt_count-tp
    total_tp, total_fp, total_fn = int(tp.sum()), int(fp.sum()), int(fn.sum())
    precision = total_tp/(total_tp+total_fp) if total_tp+total_fp else 0.0
    recall = total_tp/(total_tp+total_fn) if total_tp+total_fn else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    f2 = 5*precision*recall/(4*precision+recall) if 4*precision+recall else 0.0
    critical_gt = int(gt_count[list(CRITICAL)].sum()); critical_fn = int(fn[list(CRITICAL)].sum())
    clean = ~defective
    row = {'configuration':config_name, 'model':model_name, 'tau_critical':tau_critical, 'tau_regular':tau_regular,
           'precision':precision, 'recall':recall, 'F1':f1, 'F2':f2,
           'critical_recall':(critical_gt-critical_fn)/critical_gt if critical_gt else np.nan,
           'critical_FN_rate':critical_fn/critical_gt if critical_gt else np.nan,
           'R_small':size_tp['small']/size_gt['small'] if size_gt['small'] else np.nan,
           'R_medium':size_tp['medium']/size_gt['medium'] if size_gt['medium'] else np.nan,
           'R_large':size_tp['large']/size_gt['large'] if size_gt['large'] else np.nan,
           'defective_to_PASS_rate':int((defective & ~predicted_defective).sum())/int(defective.sum()) if defective.sum() else np.nan,
           'clean_to_DEFECTIVE_rate':int((clean & predicted_defective).sum())/int(clean.sum()) if clean.sum() else np.nan,
           'test_images':len(GT), 'clean_images':int(clean.sum()), 'GT_boxes':int(gt_count.sum())}
    class_rows = []
    for cid, cls in enumerate(CLASSES):
        class_rows.append({'configuration':config_name,'class':cls,'GT':int(gt_count[cid]),'TP':int(tp[cid]),'FP':int(fp[cid]),'FN':int(fn[cid]),
                           'precision':tp[cid]/(tp[cid]+fp[cid]) if tp[cid]+fp[cid] else 0.0,
                           'recall':tp[cid]/gt_count[cid] if gt_count[cid] else np.nan})
    return row, class_rows

config_rows, class_rows = [], []
for short, model_name in [('B0','B0'),('B1','B1')]:
    taus = models_cfg[short]
    cases = [(short, DEFAULT_CONF, DEFAULT_CONF),
             ('B2' if short=='B0' else 'Proposed', taus['tau_critical'], taus['tau_regular'])]
    for label, tc, tr in cases:
        row, per_class = evaluate_fixed_threshold(model_name, records[model_name], tc, tr, label)
        config_rows.append(row); class_rows.extend(per_class)

primary = pd.DataFrame(config_rows).set_index('configuration').loc[['B0','B1','B2','Proposed']].reset_index()
per_class = pd.DataFrame(class_rows)
display(primary.round(4))
display(per_class.pivot(index='class',columns='configuration',values='recall').reindex(CLASSES).round(4))

,configuration,model,tau_critical,tau_regular,precision,recall,F1,F2,critical_recall,critical_FN_rate,R_small,R_medium,R_large,defective_to_PASS_rate,clean_to_DEFECTIVE_rate,test_images,clean_images,GT_boxes
0,B0,B0,0.25,0.25,0.7145,0.9517,0.8162,0.8925,0.9604,0.0396,0.9548,0.9153,NaN,0.0,NaN,480,0,2424
1,B1,B1,0.25,0.25,0.7437,0.9266,0.8251,0.8831,0.9406,0.0594,0.9289,0.8995,NaN,0.0,NaN,480,0,2424
2,B2,B0,0.02,0.58,0.6303,0.9290,0.7510,0.8486,0.9765,0.0235,0.9329,0.8836,NaN,0.0,NaN,480,0,2424
3,Proposed,B1,0.01,0.57,0.6710,0.9035,0.7700,0.8449,0.9567,0.0433,0.9087,0.8413,NaN,0.0,NaN,480,0,2424


configuration,B0,B1,B2,Proposed
class,,,,
Missing_hole,0.9387,0.8456,0.9387,0.8382
Mouse_bite,0.9450,0.9500,0.9150,0.9050
Open_circuit,0.9703,0.9554,0.9827,0.9802
Short,0.9505,0.9257,0.9703,0.9332
Spur,0.9332,0.9134,0.8490,0.8342
Spurious_copper,0.9728,0.9703,0.9183,0.9307


## 4. mAP, bảng tổng hợp và lưu kết quả

mAP là metric confidence-swept, vì vậy không thay đổi theo threshold triển khai; B0/B2 dùng cùng mAP của B0, B1/Proposed dùng cùng mAP của B1. Notebook gọi evaluator Ultralytics trên đúng test split để lấy mAP cùng protocol cho cả hai checkpoint.

In [4]:
map_rows = {}
for short, checkpoint in CHECKPOINTS.items():
    model = YOLO(str(checkpoint))
    with torch.no_grad():
        result = model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                          conf=PRED_CONF, iou=NMS_IOU, max_det=300, plots=False, save_json=False,
                          project=str(OUT / 'ultralytics_runs'), name=short, exist_ok=True, verbose=True)
    map_rows[short] = {'mAP50': float(result.box.map50), 'mAP50_95': float(result.box.map)}
    del result, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()

primary['mAP50'] = primary['model'].map(lambda x: map_rows[x]['mAP50'])
primary['mAP50_95'] = primary['model'].map(lambda x: map_rows[x]['mAP50_95'])
primary['threshold_protocol'] = primary['configuration'].map({
    'B0': 'default 0.25',
    'B1': 'default 0.25',
    'B2': 'locked B0 dual threshold',
    'Proposed': 'locked B1 dual threshold'
})
primary.to_csv(OUT / 'week10_primary_metrics.csv', index=False)
per_class.to_csv(OUT / 'week10_per_class_metrics.csv', index=False)
summary = {
    'protocol': 'YOLOv8s Week 10; locked test split; no threshold tuning on test',
    'test_split_sha256': hashlib.sha256(SPLIT.read_bytes()).hexdigest(),
    'test_images': len(IMAGES),
    'imgsz': IMGSZ,
    'nms_iou': NMS_IOU,
    'matching_iou': MATCH_IOU,
    'default_conf': DEFAULT_CONF,
    'threshold_config': str(THRESHOLD_FILE.relative_to(ROOT)),
    'checkpoint_sha256': {k: models_cfg[k]['checkpoint_sha256'] for k in models_cfg},
    'map_metrics': map_rows,
    'clean_images': int(sum(not g for g in GT)),
    'caveat': 'B0 test results existed before Week 10; B0 batch 16 and B1 batch 8.'
}
(OUT / 'week10_protocol_and_metrics.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('\n🏆 BẢNG TỔNG HỢP SO SÁNH B0, B1, B2, PROPOSED (Test 480 ảnh):')
display(primary.round(4))
print('\n📁 Kết quả đã lưu tại:', OUT)

# Đóng gói ZIP tự động nếu chạy trên Kaggle để tiện tải về máy
if IS_KAGGLE:
    out_zip = ROOT / 'B0_B1_B2_Proposed_results.zip'
    with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in OUT.rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(ROOT))
    print(f'\n📦 Đã đóng gói toàn bộ kết quả vào: {out_zip} ({round(out_zip.stat().st_size / 1024, 1)} KB)')
    print('👉 Bấm vào link dưới đây để tải file zip về máy:')
    display(FileLink(out_zip.name))


Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 168 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 18.3MB/s]
val: Scanning /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)/train/labels... 480 images, 0 backgrounds, 0 corrupt: 100%|██████████| 480/480 [00:00<00:00, 765.95it/s]


val: WARNING ⚠️ Cache directory /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)/train is not writeable, cache not saved.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 240/240 [00:20<00:00, 11.84it/s]


                   all        480       2424      0.844       0.92      0.878      0.422
          Missing_hole         80        408      0.864      0.939      0.883      0.336
            Mouse_bite         80        400      0.992      0.928      0.968      0.552
          Open_circuit         80        404      0.977       0.94      0.975      0.512
                 Short         80        404      0.643      0.926      0.884      0.344
                  Spur         80        404      0.991      0.855      0.967      0.458
       Spurious_copper         80        404      0.598      0.933      0.592      0.332
Speed: 0.6ms preprocess, 24.5ms inference, 0.0ms loss, 1.5ms postprocess per image
Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8s-p2 summary (fused): 207 layers, 10,628,008 parameters, 0 gradients, 36.6 GFLOPs


val: Scanning /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)/train/labels... 480 images, 0 backgrounds, 0 corrupt: 100%|██████████| 480/480 [00:00<00:00, 736.77it/s]


... (3 outputs left out)


🏆 BẢNG TỔNG HỢP SO SÁNH B0, B1, B2, PROPOSED (Test 480 ảnh):


,configuration,model,tau_critical,tau_regular,precision,recall,F1,F2,critical_recall,critical_FN_rate,...,R_medium,R_large,defective_to_PASS_rate,clean_to_DEFECTIVE_rate,test_images,clean_images,GT_boxes,mAP50,mAP50_95,threshold_protocol
0,B0,B0,0.25,0.25,0.7145,0.9517,0.8162,0.8925,0.9604,0.0396,...,0.9153,NaN,0.0,NaN,480,0,2424,0.8782,0.4222,default 0.25
1,B1,B1,0.25,0.25,0.7437,0.9266,0.8251,0.8831,0.9406,0.0594,...,0.8995,NaN,0.0,NaN,480,0,2424,0.8508,0.3999,default 0.25
2,B2,B0,0.02,0.58,0.6303,0.9290,0.7510,0.8486,0.9765,0.0235,...,0.8836,NaN,0.0,NaN,480,0,2424,0.8782,0.4222,locked B0 dual threshold
3,Proposed,B1,0.01,0.57,0.6710,0.9035,0.7700,0.8449,0.9567,0.0433,...,0.8413,NaN,0.0,NaN,480,0,2424,0.8508,0.3999,locked B1 dual threshold



📁 Kết quả đã lưu tại: /kaggle/working/evaluation/B0_B1_yolov8s_week10

📦 Đã đóng gói toàn bộ kết quả vào: /kaggle/working/B0_B1_B2_Proposed_results.zip (4.4 KB)
👉 Bấm vào link dưới đây để tải file zip về máy:


/kaggle/working/B0_B1_B2_Proposed_results.zip

## 5. Cách kết luận

Không mặc định Proposed tốt nhất. So sánh B0→B1 ở ngưỡng mặc định để mô tả khác biệt B0/B1; B0→B2 và B1→Proposed mô tả tác động threshold trên từng checkpoint. Ghi nhận B0 dùng batch 16 còn B1 batch 8. Với `clean_images=0`, Board false-reject (clean→DEFECTIVE) là N/A; không thay bằng 0. Sau khi xem bảng này không điều chỉnh threshold và không chạy lại test để tìm cấu hình tốt hơn.